# 迭代器与生成器

学习目标：实现可重复遍历的数据源，逐步驱动生成器，并正确结束惰性迭代与辅助方法链。

前置知识：对象方法、Symbol、for...of、函数闭包、try/catch/finally 与数组转换。

适用版本：ECMAScript 2025；Node.js 24.11.0 原生支持 Iterator.from 及本章使用的迭代器辅助方法。异步迭代在 async/await 章节展开。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/javascript。以下命令均从此目录运行；每个入口使用独立 Node.js 进程。

配套脚本：位于 scripts/20-iterators-and-generators/。

1. [iterable-protocol.mjs](scripts/20-iterators-and-generators/iterable-protocol.mjs)：手写可重复遍历的整数区间。
2. [iterator-result-error.mjs](scripts/20-iterators-and-generators/iterator-result-error.mjs)：next 返回原始值违反协议。
3. [consumption.mjs](scripts/20-iterators-and-generators/consumption.mjs)：容器与游标的重复使用。
4. [generator-basics.mjs](scripts/20-iterators-and-generators/generator-basics.mjs)：暂停、结束值与重新创建。
5. [generator-input.mjs](scripts/20-iterators-and-generators/generator-input.mjs)：向暂停的 yield 传值。
6. [generator-throw.mjs](scripts/20-iterators-and-generators/generator-throw.mjs)：在生成器内部接住注入异常。
7. [generator-throw-error.mjs](scripts/20-iterators-and-generators/generator-throw-error.mjs)：未捕获的注入异常会传播。
8. [early-close.mjs](scripts/20-iterators-and-generators/early-close.mjs)：break 与手动消费的结束责任。
9. [return-through-finally.mjs](scripts/20-iterators-and-generators/return-through-finally.mjs)：结束请求经过可暂停的 finally。
10. [delegation.mjs](scripts/20-iterators-and-generators/delegation.mjs)：委托逐项值并读取内部结束值。
11. [delegation-close.mjs](scripts/20-iterators-and-generators/delegation-close.mjs)：委托链上的结束与清理顺序。
12. [lazy-helpers.mjs](scripts/20-iterators-and-generators/lazy-helpers.mjs)：惰性映射、筛选、限制和消费。
13. [flat-map-error.mjs](scripts/20-iterators-and-generators/flat-map-error.mjs)：flatMap 拒绝原始字符串结果。
14. [terminal-helpers.mjs](scripts/20-iterators-and-generators/terminal-helpers.mjs)：终结方法与空源规则。
15. [helper-close.mjs](scripts/20-iterators-and-generators/helper-close.mjs)：短路关闭生成器而非留下游标。
16. [empty-reduce-error.mjs](scripts/20-iterators-and-generators/empty-reduce-error.mjs)：空迭代器的 reduce 需要初始值。

## 1 可迭代协议与迭代器协议

可迭代对象（iterable）回答“怎样开始遍历”：它的 Symbol.iterator 方法返回迭代器。迭代器（iterator）回答“下一个是什么”：它的 next 方法返回结果对象。二者是接口约定，不要求继承某个类；一个对象也可以同时满足两种协议。

结果对象的 value 是本次值，done 为 true 表示结束。undefined 也可以是有效元素，不能用 value 是否为 undefined 判断结束。next 必须返回对象；缺少 done 按 false 处理，通常仍建议明确写布尔状态。自定义迭代器应在结束后持续返回 done: true，普通对象不会被引擎自动替你维持该约定。

下例每次调用 Symbol.iterator 都建立独立计数器，因此 range 可重复遍历；单个 iterator 则保存自己的进度。上界 end 不包含在输出内。

[iterable-protocol.mjs](scripts/20-iterators-and-generators/iterable-protocol.mjs)：

```javascript
function range(start, end) {
  return {
    [Symbol.iterator]() {
      // 计数器属于这次创建的迭代器，两次遍历不会共用进度。
      let current = start;
      return {
        next() {
          // 结束以 done 判断；上界 end 不作为有效元素返回。
          if (current >= end) return { value: undefined, done: true };
          return { value: current++, done: false };
        }
      };
    }
  };
}
const values = range(2, 5);
// 保留一个手动迭代器，再创建新迭代器展开；比较两者各自的进度。
const iterator = values[Symbol.iterator]();
console.log(JSON.stringify(iterator.next()));
console.log([...values].join(","), [...values].join(","));
console.log(JSON.stringify(iterator.next()));
console.log(JSON.stringify(iterator.next()));
console.log(iterator.next().done, iterator.next().done);

// 按本例输入运行，输出依次为：
// {"value":2,"done":false}
// 2,3,4 2,3,4
// {"value":3,"done":false}
// {"value":4,"done":false}
// true true
```

Step 1：运行本节示例。

```bash
node scripts/20-iterators-and-generators/iterable-protocol.mjs
```

[iterator-result-error.mjs](scripts/20-iterators-and-generators/iterator-result-error.mjs)：

```javascript
const broken = {
  [Symbol.iterator]() { return this; },
  next() { return 7; }
};
console.log([...broken]);

// 独立运行：退出状态为 1；诊断包含 TypeError；Iterator result 7 is not an object。
```

Step 1：单独运行反例，预期非零退出。

```bash
node scripts/20-iterators-and-generators/iterator-result-error.mjs
```

## 2 谁会消费迭代器，谁能重头开始

for...of、数组展开、数组解构和许多集合构造器通过可迭代协议获取元素；对象属性枚举与对象展开不是这套协议。数组自身可重复创建新的迭代器，而数组迭代器自身的 Symbol.iterator 返回自己，第二次使用同一个已耗尽迭代器不会自动重置。

只有 next 的对象可以满足迭代器协议，但未必可供 for...of 使用。反过来，可迭代容器未必要有 next；上一节 range 就将容器和游标分开。需要重复计算时保留能创建新迭代器的工厂，不要共享同一个已经前进的游标。

[consumption.mjs](scripts/20-iterators-and-generators/consumption.mjs)：

```javascript
const values = [undefined, "JS"];
const iterator = values.values();
console.log(iterator[Symbol.iterator]() === iterator);
const first = iterator.next();
console.log(first.value, first.done);
console.log([...iterator].join(","));
console.log([...iterator].length, [...values].length);
console.log([...new Set(["A", "A", "B"])].join(","));
const iteratorOnly = { next() { return { done: true }; } };
console.log(typeof iteratorOnly.next, typeof iteratorOnly[Symbol.iterator]);

// 按本例输入运行，输出依次为：
// true
// undefined false
// JS
// 0 2
// A,B
// function undefined
```

Step 1：运行本节示例。

```bash
node scripts/20-iterators-and-generators/consumption.mjs
```

## 3 生成器用暂停点保存执行状态

function* 定义生成器函数，调用后得到生成器对象，函数体先暂停，直到 next 才开始执行。yield 给出一个值并暂停，下一次 next 从暂停处恢复。局部变量和控制流随对象保存，不需要手写状态计数。

生成器对象同时是可迭代对象和迭代器。yield 得到的结果对象通常 done: false；return 则给出 done: true 的结束值。for...of 与展开只收集 done: false 的元素，不包含最终 return 值。生成器完成后不会重新执行，再次调用生成器函数才是创建新的遍历。

[generator-basics.mjs](scripts/20-iterators-and-generators/generator-basics.mjs)：

```javascript
const events = [];
function* lessons() {
  events.push("开始");
  yield "Map";
  yield "Set";
  return "完成";
}
const iterator = lessons();
console.log(events.length);
for (let index = 0; index < 4; index += 1) {
  const step = iterator.next();
  console.log(step.value, step.done);
}
console.log(events.join(","));
console.log([...lessons()].join(","));

// 按本例输入运行，输出依次为：
// 0
// Map false
// Set false
// 完成 true
// undefined true
// 开始
// Map,Set
```

Step 1：运行本节示例。

```bash
node scripts/20-iterators-and-generators/generator-basics.mjs
```

## 4 next 的输入与 throw 的异常注入

next(value) 的实参会成为上一个暂停的 yield 表达式的结果。第一次 next 的实参被忽略，因为此时还没有等待接收值的 yield；初始化数据应通过生成器函数参数传入。for...of 不向 next 传值，双向交互应由调用者显式驱动。

throw(error) 在暂停位置注入异常，生成器内部可以 catch 并继续 yield；未捕获时异常传播到调用方，生成器完成。这里是生成器的 throw 方法，不是给迭代器对象增加一个错误属性；一般迭代器的 throw 是可选的，不能假定所有迭代器都提供。

[generator-input.mjs](scripts/20-iterators-and-generators/generator-input.mjs)：

```javascript
function* multiply(base) {
  const factor = yield "请输入倍数";
  return base * factor;
}
const iterator = multiply(4);
console.log(JSON.stringify(iterator.next(999)));
console.log(JSON.stringify(iterator.next(3)));

// 按本例输入运行，输出依次为：
// {"value":"请输入倍数","done":false}
// {"value":12,"done":true}
```

Step 1：运行本节示例。

```bash
node scripts/20-iterators-and-generators/generator-input.mjs
```

[generator-throw.mjs](scripts/20-iterators-and-generators/generator-throw.mjs)：

```javascript
function* session() {
  try {
    yield "准备";
  } catch (error) {
    if (!(error instanceof RangeError)) throw error;
    yield "恢复:" + error.message;
  }
  return "结束";
}
const iterator = session();
console.log(iterator.next().value);
console.log(iterator.throw(new RangeError("输入越界")).value);
console.log(JSON.stringify(iterator.next()));

// 按本例输入运行，输出依次为：
// 准备
// 恢复:输入越界
// {"value":"结束","done":true}
```

Step 1：运行本节示例。

```bash
node scripts/20-iterators-and-generators/generator-throw.mjs
```

[generator-throw-error.mjs](scripts/20-iterators-and-generators/generator-throw-error.mjs)：

```javascript
function* session() {
  try { yield "准备"; }
  finally { console.log("清理已执行"); }
}
const iterator = session();
iterator.next();
iterator.throw(new Error("停止学习"));

// 独立运行：退出状态为 1；诊断包含 清理已执行；Error: 停止学习。
```

Step 1：单独运行反例，预期非零退出。

```bash
node scripts/20-iterators-and-generators/generator-throw-error.mjs
```

## 5 return、提前退出与清理

return(value) 请求生成器从暂停位置结束，控制流仍会经过已进入的 try 对应的 finally。for...of 因 break、循环体 throw 或函数 return 提前离开时，会尝试调用迭代器的 return；普通同循环 continue 不结束遍历。自然耗尽时通常不额外调用 return，生成器自身运行到结尾仍会执行 finally。

只停止手动 next 并不会自动执行 finally，也不能依靠垃圾回收完成它。下例记录清理事件，手动消费方用 finally 调用 return，确保自己结束已启动的生成器。真实资源还应在该清理位置调用相应宿主的关闭操作。若生成器从未启动就 return，函数体和其中 finally 都不会执行，因此资源获取宜放在生成器体内。

[early-close.mjs](scripts/20-iterators-and-generators/early-close.mjs)：

```javascript
const events = [];
function* items(label) {
  try {
    yield 1;
    yield 2;
  } finally {
    events.push("关闭:" + label);
  }
}
// for...of 的 break 会请求关闭迭代器，触发已进入的 finally。
for (const value of items("循环")) {
  console.log(value);
  break;
}
// 手动 next 的调用方负责结束迭代；这里只停止读取还不足以触发清理。
const manual = items("手动");
try {
  console.log(manual.next().value);
  console.log(events.join(","));
} finally {
  manual.return();
}
// 从未 next 的生成器尚未进入函数体，所以 return 不执行这里的 finally。
const neverStarted = items("未启动");
neverStarted.return();
console.log(events.join(","), manual.next().done);

// 按本例输入运行，输出依次为：
// 1
// 1
// 关闭:循环
// 关闭:循环,关闭:手动 true
```

Step 1：运行本节示例。

```bash
node scripts/20-iterators-and-generators/early-close.mjs
```

## 6 finally 中再次 yield 的边界

return 是结束请求，不代表调用一瞬间必然已全部清理。若 finally 内含 yield，该 yield 仍会暂停并返回 done: false，继续 next 后才完成原来的 return；finally 的新 return 或 throw 也可能覆盖原请求。

这对资源生命周期很重要：for...of 的提前关闭只调用一次 return，不会为你循环驱动 finally 内更多 yield。不要在负责释放关键资源的 finally 中安排需要额外消费才能完成的步骤。本例显式驱动到 done: true，观察这个边界而不遗留暂停状态。

[return-through-finally.mjs](scripts/20-iterators-and-generators/return-through-finally.mjs)：

```javascript
function* values() {
  try { yield 1; }
  finally { yield "最后一步"; }
}
const iterator = values();
console.log(JSON.stringify(iterator.next()));
console.log(JSON.stringify(iterator.return("结束值")));
console.log(JSON.stringify(iterator.next()));
console.log(iterator.next().done);

// 按本例输入运行，输出依次为：
// {"value":1,"done":false}
// {"value":"最后一步","done":false}
// {"value":"结束值","done":true}
// true
```

Step 1：运行本节示例。

```bash
node scripts/20-iterators-and-generators/return-through-finally.mjs
```

## 7 yield* 委托给另一段遍历

yield* 消费另一个可迭代对象，把它的逐项值委托给外层消费方。委托结束时，yield* 表达式的值是内部迭代器的结束值；因此既可拼接数组等序列，也可从子生成器取得汇总结果。

委托还会转发 next 的输入以及适用的 return、throw 请求，并非简单复制一个数组。内层没有 throw 方法时，外层注入异常会先尝试关闭内层再抛 TypeError；有方法也必须遵守结果对象协议。下面提前 return 的例子观察到先经过内层 finally，再经过外层 finally。

[delegation.mjs](scripts/20-iterators-and-generators/delegation.mjs)：

```javascript
function* group() {
  yield "Map";
  yield "Set";
  return 2;
}
function* course() {
  yield* ["开始"];
  const count = yield* group();
  yield "已学:" + count;
}
console.log([...course()].join(","));

// 按本例输入运行，输出依次为：
// 开始,Map,Set,已学:2
```

Step 1：运行本节示例。

```bash
node scripts/20-iterators-and-generators/delegation.mjs
```

[delegation-close.mjs](scripts/20-iterators-and-generators/delegation-close.mjs)：

```javascript
const events = [];
function* inner() {
  try { yield 1; yield 2; }
  finally { events.push("内层"); }
}
function* outer() {
  try { yield* inner(); }
  finally { events.push("外层"); }
}
const iterator = outer();
console.log(iterator.next().value);
console.log(JSON.stringify(iterator.return("停止")));
console.log(events.join(","), iterator.next().done);

// 按本例输入运行，输出依次为：
// 1
// {"value":"停止","done":true}
// 内层,外层 true
```

Step 1：运行本节示例。

```bash
node scripts/20-iterators-and-generators/delegation-close.mjs
```

## 8 Iterator.from 与惰性辅助方法

ECMAScript 2025 的迭代器辅助方法在 Iterator.prototype 上，Node.js 24.11.0 原生提供；它们不意味着 Array 本身多了一套惰性方法。数组迭代器和生成器继承这些方法，自定义普通迭代器可用 Iterator.from 包装。Iterator.from 接受可迭代对象或迭代器，不会把数据全部复制成数组；它也不重置已消费的进度。Iterator 设计为可派生的基类，不能直接 new Iterator() 得到有数据的游标。

| 原文名称 | 中文名称／含义 | 惰性行为 |
| --- | --- | --- |
| map | 映射 | 按需求值并转换每个元素 |
| filter | 筛选 | 拉取到满足条件的元素 |
| flatMap | 映射后展开 | 顺序展开回调返回的可迭代对象或迭代器对象 |
| take | 限制数量 | 最多给出指定数量的元素 |
| drop | 跳过开头 | 跳过指定数量后继续 |

这些方法返回新的辅助迭代器，真正拉取发生在 next 或终结操作中。回调一般收到元素与从 0 起的索引，不像数组方法那样再提供原数组参数。take/drop 会将 limit 转成整数，NaN 或转换后为负数时抛 RangeError；教学调用直接提供非负整数。flatMap 不接受原始字符串作为回调结果，需要时先显式转成字符数组。

下例自定义数据源只在 next 内增加 pulled，建立链后仍为 0，取两个结果只计算前三个输入。需要筛选才能结束的无限源仍可能一直找不到匹配值；惰性不等于自动有限。

[lazy-helpers.mjs](scripts/20-iterators-and-generators/lazy-helpers.mjs)：

```javascript
let pulled = 0;
let closed = false;
// pulled 统计实际拉取次数，closed 保存数据源被关闭后的状态。
const source = {
  next() {
    if (closed) return { done: true };
    pulled += 1;
    return { value: pulled, done: false };
  },
  return() {
    closed = true;
    console.log("数据源关闭");
    return { done: true };
  }
};
// 建链时不取数据；toArray 才逐个拉取，take 收到两个结果后关闭源。
const selected = Iterator.from(source)
  .map(value => value * 10)
  .filter(value => value >= 20)
  .take(2);
console.log(pulled);
console.log(selected.toArray().join(","), pulled);
// 已消费的迭代器不重置；再次收集只能得到空结果。
console.log(selected.toArray().length);
console.log(Iterator.from([1, 2, 3]).drop(1)
  .flatMap(value => [value, -value]).toArray().join(","));

// 按本例输入运行，输出依次为：
// 0
// 数据源关闭
// 20,30 3
// 0
// 2,-2,3,-3
```

Step 1：运行本节示例。

```bash
node scripts/20-iterators-and-generators/lazy-helpers.mjs
```

[flat-map-error.mjs](scripts/20-iterators-and-generators/flat-map-error.mjs)：

```javascript
Iterator.from([1]).flatMap(() => "JS").toArray();

// 独立运行：退出状态为 1；诊断包含 TypeError；called on non-object。
```

Step 1：单独运行反例，预期非零退出。

```bash
node scripts/20-iterators-and-generators/flat-map-error.mjs
```

## 9 终结操作、短路与剩余进度

终结操作会立即消费当前进度；toArray 和 reduce 等需要走到末尾，不能直接用于未加限制的无限源。辅助链是一次遍历的状态，不是可以无限重复运行的数据管道。

| 原文名称 | 中文名称／含义 | 返回值与边界 |
| --- | --- | --- |
| toArray | 收集成数组 | 消费剩余元素并分配数组 |
| reduce | 归并 | 返回累积结果；空源且无初值会抛 TypeError |
| forEach | 逐项处理 | 用于副作用，返回 undefined |
| some | 存在判断 | 找到真值条件即短路；空源为 false |
| every | 全称判断 | 遇到假值条件即短路；空源为 true |
| find | 查找元素 | 找到即短路；未找到返回 undefined |

some/every/find 短路时会关闭底层迭代器，take 到数量上限且继续请求结束时也会关闭底层。回调抛错同样需要按方法的关闭规则传播，不要假定短路以后生成器还能接着取。以下例子分别展示终结结果，以及 find 找到第一个元素后立即执行生成器清理。

[terminal-helpers.mjs](scripts/20-iterators-and-generators/terminal-helpers.mjs)：

```javascript
console.log(Iterator.from([2, 3, 4]).reduce((sum, value) => sum + value, 0));
console.log(Iterator.from([]).reduce((sum, value) => sum + value, 10));
const visited = [];
const result = Iterator.from(["Map", "Set"]).forEach((value, index) =>
  visited.push(String(index) + ":" + value));
console.log(visited.join(","), result);
console.log(Iterator.from([1, 2]).some(value => value % 2 === 0));
console.log(Iterator.from([1, 2]).every(value => value > 0));
console.log(Iterator.from([]).some(Boolean), Iterator.from([]).every(Boolean));
console.log(Iterator.from([1, 2]).find(value => value > 9));

// 按本例输入运行，输出依次为：
// 9
// 10
// 0:Map,1:Set undefined
// true
// true
// false true
// undefined
```

Step 1：运行本节示例。

```bash
node scripts/20-iterators-and-generators/terminal-helpers.mjs
```

[helper-close.mjs](scripts/20-iterators-and-generators/helper-close.mjs)：

```javascript
const events = [];
function* values() {
  try {
    events.push("拉取1"); yield 1;
    events.push("拉取2"); yield 2;
  } finally {
    events.push("关闭");
  }
}
const iterator = values();
console.log(iterator.find(value => value === 1));
console.log(events.join(","), iterator.next().done);

// 按本例输入运行，输出依次为：
// 1
// 拉取1,关闭 true
```

Step 1：运行本节示例。

```bash
node scripts/20-iterators-and-generators/helper-close.mjs
```

[empty-reduce-error.mjs](scripts/20-iterators-and-generators/empty-reduce-error.mjs)：

```javascript
Iterator.from([]).reduce((sum, value) => sum + value);

// 独立运行：退出状态为 1；诊断包含 TypeError；Reduce of a done iterator with no initial value。
```

Step 1：单独运行反例，预期非零退出。

```bash
node scripts/20-iterators-and-generators/empty-reduce-error.mjs
```

## 本章小结

- 可迭代对象负责创建游标，迭代器负责逐步提供结果；结束状态看 done。
- 生成器以 yield 保存暂停点，next、return、throw 参与恢复和结束控制流。
- 提前退出需要明确清理责任，finally 内的 yield 会延后真正完成。
- 辅助方法按需拉取，终结操作消耗进度，短路也可能关闭底层。

## 练习

1. 写一个每次能重新遍历的倒计时对象，输出 3、2、1。可核对标准：两次展开结果相同，两个独立迭代器互不影响，结束后 next 始终 done: true。
2. 用生成器先 yield 一个问题，再通过 next 接收答案。可核对标准：第一次 next 实参不作为答案，第二次实参进入上一个 yield，最终返回对象 done: true。
3. 从递增数据源筛选偶数、平方、只取前三项。可核对标准：结果为 4、16、36，建立链时不拉取，消费结束后底层清理恰好一次。
4. 比较 break、手动停止 next 和显式 return。可核对标准：记录 finally 是否执行，解释为何不能依赖“以后可能被垃圾回收”；所有创建并启动的生成器最终完成。

## 参考与引用来源

- TC39 官方 ECMAScript 2025 分页版：[§27.1.1 Iterable、Iterator 与结果协议](https://tc39.es/ecma262/2025/multipage/control-abstraction-objects.html#sec-common-iteration-interfaces)；[§27.1.3.2.1 Iterator.from](https://tc39.es/ecma262/2025/multipage/control-abstraction-objects.html#sec-iterator.from)；[§27.1.4 辅助与终结方法](https://tc39.es/ecma262/2025/multipage/control-abstraction-objects.html#sec-%iterator.prototype%-object)；[§27.5 生成器状态及 next/return/throw](https://tc39.es/ecma262/2025/multipage/control-abstraction-objects.html#sec-generator-objects)；[§15.5.5 yield 与 yield*](https://tc39.es/ecma262/2025/multipage/ecmascript-language-functions-and-classes.html#sec-generator-function-definitions-runtime-semantics-evaluation)；[§7.4.11 IteratorClose](https://tc39.es/ecma262/2025/multipage/abstract-operations.html#sec-iteratorclose)。
- MDN 用法对照：[辅助方法、惰性与终结操作](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/Iterator)；[finally 再次 yield 的结束边界](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/Generator/return)。